# M59g — collect fixed inputs for the precision audit
**Revision: M59g-2026-09-18-r1. Run all three code cells, top to bottom, on a CPU runtime.**

No GPU, training, upstream clone, or CUDA build is needed. This collects 16 preselected Chen validation inputs from your existing KITTI files and verifies that input 000001 exactly matches the frozen M58 reference. It does not evaluate AP or change any threshold. Core ML execution happens later on the Mac.

Required in Drive: original M58 export gate, `MonoDGP_M58_fixed_fp32.pt`, `m58_reference_io.npz`, canonical `kitti_chen/val.txt`, and KITTI `training/image_2` plus `training/calib`. No labels or new checkpoints are needed. Return the ZIP printed by the final cell. On session restart rerun these same three cells.

In [ ]:
# M59g-2026-09-18-r1 — all imports, paths, and logging are defined here.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, os, shlex, subprocess, sys, zipfile
REVISION = 'M59g-2026-09-18-r1'
MOBILE_REPO = Path('/content/mobile_adas3d')
MYDRIVE = Path('/content/drive/MyDrive')
M58_DIR = MYDRIVE/'mobile_adas3d_outputs/compression/monodgp_m58_coreml_conversion'
SPLIT_FILE = MYDRIVE/'mobile_adas3d_splits/kitti_chen/val.txt'
OUTPUT_ROOT = MYDRIVE/'mobile_adas3d_outputs/compression/monodgp_m59g_precision_audit'
# Change only this dataset location if your original KITTI folder is elsewhere.
DRIVE_KITTI = MYDRIVE/'datasets/kitti'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
def run_logged(command, log_path, cwd=None):
    command = [str(x) for x in command]
    print('+', shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=30)
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        code = process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n' + '\n'.join(tail))
print(REVISION, 'CPU input collection only; no model training.')


In [ ]:
# Sync the project main branch, without resetting or deleting existing changes.
REPO_URL = '/'.join(('https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'))
LOGS = OUTPUT_ROOT/'colab_logs'
if not MOBILE_REPO.exists():
    run_logged(['git','clone','--branch','main',REPO_URL,MOBILE_REPO], LOGS/'clone.log')
if not (MOBILE_REPO/'.git').exists(): raise RuntimeError(f'Not a Git checkout: {MOBILE_REPO}')
branch = subprocess.check_output(['git','branch','--show-current'], cwd=MOBILE_REPO, text=True).strip()
dirty = subprocess.check_output(['git','status','--porcelain'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main' or dirty: raise RuntimeError(f'Expected clean main checkout; branch={branch}. Preserve local edits before syncing.\n{dirty}')
run_logged(['git','pull','--ff-only'], LOGS/'sync.log', MOBILE_REPO)
SCRIPT = MOBILE_REPO/'scripts/collect_monodgp_m59g_inputs.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'Updated main must contain {SCRIPT}')
run_logged([sys.executable,'-m','pip','install','-q','numpy>=2.0,<2.4','Pillow','opencv-python-headless'], LOGS/'dependencies.log')
print('Project commit:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=MOBILE_REPO, text=True).strip())
print(REVISION, 'setup complete')


In [ ]:
# Capture only the 16 predetermined inputs. Fail closed if the anchor differs.
for path in (SPLIT_FILE, M58_DIR/'m58_coreml_export_gate.json', M58_DIR/'MonoDGP_M58_fixed_fp32.pt', M58_DIR/'m58_reference_io.npz'):
    if not path.is_file(): raise FileNotFoundError(f'Required original artifact missing: {path}')
sys.path.insert(0, str(MOBILE_REPO))
from scripts.collect_monodgp_m59g_inputs import choose_samples
_, SAMPLE_IDS = choose_samples(SPLIT_FILE)
roots = [Path('/content/monodgp_kitti_m56d'), Path('/content/kitti'), DRIVE_KITTI]
def find_directory(names, suffix):
    candidates = [root/'training'/name for root in roots for name in names]
    for folder in candidates:
        if all((folder/(sample_id+suffix)).is_file() for sample_id in SAMPLE_IDS): return folder
    raise FileNotFoundError('No directory contains all 16 selected files. Check DRIVE_KITTI in cell 1. Tried: ' + ', '.join(map(str,candidates)))
IMAGE_DIR = find_directory(('image_2','image_02'), '.png')
CALIB_DIR = find_directory(('calib',), '.txt')
print('Images:', IMAGE_DIR, '\nCalibration:', CALIB_DIR, '\nSelected IDs:', SAMPLE_IDS)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
BUNDLE = OUTPUT_ROOT/('m59g_inputs_'+stamp)
run_logged([sys.executable,'-u',SCRIPT,'--image-dir',IMAGE_DIR,'--calibration-dir',CALIB_DIR,'--split-file',SPLIT_FILE,'--m58-dir',M58_DIR,'--output-dir',BUNDLE], LOGS/(stamp+'_collect.log'), MOBILE_REPO)
manifest = json.loads((BUNDLE/'m59g_input_manifest.json').read_text())
assert manifest['complete'] and manifest['sample_count'] == 16 and manifest['anchor_preprocessing_bit_exact']
ZIP = BUNDLE.with_suffix('.zip')
with zipfile.ZipFile(ZIP) as archive: assert archive.testzip() is None
print('DONE. Return this ZIP:', ZIP)
print('Size (MB):', round(ZIP.stat().st_size/1e6, 2))
print('No training, no AP evaluation, no threshold change. Mac audit is next.')
